In [18]:
!pip install -q streamlit pyngrok transformers torch scikit-learn pandas numpy
print("✅ All packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 107.2 MB/s eta 0:00:00
✅ All packages installed


In [19]:
from google.colab import files
print("Upload train.tsv, valid.tsv, test.tsv")
uploaded = files.upload()
print("✅ Uploaded:", list(uploaded.keys()))


Upload train.tsv, valid.tsv, test.tsv


Saving test.tsv to test.tsv
Saving train.tsv to train.tsv
Saving valid.tsv to valid.tsv
✅ Uploaded: ['test.tsv', 'train.tsv', 'valid.tsv']


In [23]:
import os
import glob

# Search everywhere for TSV files
tsv_files = glob.glob('/content/**/*.tsv', recursive=True)
print("Found TSV files:")
for f in tsv_files:
    print(f"  {f}")

# Also check root content folder
print("\nAll files in /content:")
for f in os.listdir('/content'):
    print(f"  {f}")

Found TSV files:
  /content/mlops-misinformation-detection/test.tsv
  /content/mlops-misinformation-detection/valid.tsv
  /content/mlops-misinformation-detection/train.tsv
  /content/mlops-misinformation-detection/data/raw/test.tsv
  /content/mlops-misinformation-detection/data/raw/valid.tsv
  /content/mlops-misinformation-detection/data/raw/train.tsv

All files in /content:
  .config
  mlops-misinformation-detection
  misinformation_app
  sample_data


In [22]:
import os
files = os.listdir('/content')
tsv_files = [f for f in files if '.tsv' in f]
print("TSV files found:", tsv_files)

TSV files found: []


In [27]:
import os

os.makedirs('/content/mlops-misinformation-detection/data/raw', exist_ok=True)
os.makedirs('/content/mlops-misinformation-detection/src', exist_ok=True)

import shutil
shutil.copy('/content/mlops-misinformation-detection/train.tsv', '/content/mlops-misinformation-detection/data/raw/train.tsv')
shutil.copy('/content/mlops-misinformation-detection/valid.tsv', '/content/mlops-misinformation-detection/data/raw/valid.tsv')
shutil.copy('/content/mlops-misinformation-detection/test.tsv',  '/content/mlops-misinformation-detection/data/raw/test.tsv')

print("✅ Folders created and files copied")
print(os.listdir('/content/mlops-misinformation-detection/data/raw'))

✅ Folders created and files copied
['test.tsv', '.gitkeep', 'valid.tsv.dvc', 'valid.tsv', 'train.tsv']


In [28]:
import os
import shutil

# Files are already in the repo - just make sure folders exist
os.makedirs('/content/misinformation_app/data/raw', exist_ok=True)
os.makedirs('/content/misinformation_app/src', exist_ok=True)

# Copy from repo folder to app folder
shutil.copy('/content/mlops-misinformation-detection/data/raw/train.tsv',
            '/content/misinformation_app/data/raw/train.tsv')
shutil.copy('/content/mlops-misinformation-detection/data/raw/valid.tsv',
            '/content/misinformation_app/data/raw/valid.tsv')
shutil.copy('/content/mlops-misinformation-detection/data/raw/test.tsv',
            '/content/misinformation_app/data/raw/test.tsv')

print("✅ Files copied:")
print(os.listdir('/content/misinformation_app/data/raw'))

✅ Files copied:
['test.tsv', 'valid.tsv', 'train.tsv']


In [30]:
import os

app = """
import streamlit as st
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer

st.set_page_config(page_title="Misinformation Detector", page_icon="🔍")

st.title("🔍 Misinformation Detection System")
st.markdown("**MSc Dissertation — 7005SCN | Coventry University**")
st.divider()

COLUMNS = ["id","label","statement","subject","speaker","job_title",
           "state_info","party_affiliation","barely_true_count","false_count",
           "half_true_count","mostly_true_count","pants_fire_count","context"]
FAKE_LABELS = {"false","barely-true","pants-fire"}

@st.cache_data
def load_data():
    df = pd.read_csv("data/raw/train.tsv", sep="\\t", header=None, names=COLUMNS)
    df["binary_label"] = df["label"].apply(lambda x: 1 if x in FAKE_LABELS else 0)
    return df

@st.cache_resource
def train_models():
    df = load_data()
    tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
    X = tfidf.fit_transform(df["statement"].tolist())
    y = df["binary_label"].tolist()
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X, y)
    nb = MultinomialNB()
    nb.fit(X, y)
    return tfidf, lr, nb

st.sidebar.header("Model Performance")
st.sidebar.markdown('''
| Model | F1 Macro |
|-------|----------|
| RoBERTa-base | 0.629 |
| Logistic Regression | 0.592 |
| Naive Bayes | 0.587 |
''')
st.sidebar.divider()
st.sidebar.markdown("MLOps Stack: DVC, MLflow, GitHub Actions, Docker, Streamlit")

model_choice = st.selectbox("Select model:",
    ["Logistic Regression", "Naive Bayes", "All Models"])

examples = [
    "Select an example...",
    "The unemployment rate fell to its lowest level in 50 years.",
    "We have created more jobs than any president in history.",
    "The government is adding chemicals to the water supply.",
    "This country has the highest taxes in the world.",
]
example = st.selectbox("Or pick an example:", examples)
user_input = st.text_area("Enter statement:",
    value="" if example == "Select an example..." else example,
    height=100)

if st.button("Classify Statement", type="primary", use_container_width=True):
    if not user_input.strip():
        st.warning("Please enter a statement.")
    else:
        with st.spinner("Analysing..."):
            tfidf, lr, nb = train_models()
        st.divider()
        st.subheader("Results")
        results = {}
        if model_choice in ["Logistic Regression", "All Models"]:
            X = tfidf.transform([user_input])
            pred = lr.predict(X)[0]
            conf = max(lr.predict_proba(X)[0])
            results["Logistic Regression"] = (pred, conf)
        if model_choice in ["Naive Bayes", "All Models"]:
            X = tfidf.transform([user_input])
            pred = nb.predict(X)[0]
            conf = max(nb.predict_proba(X)[0])
            results["Naive Bayes"] = (pred, conf)
        for name, (pred, conf) in results.items():
            label = "REAL" if pred == 0 else "FAKE"
            color = "green" if pred == 0 else "red"
            st.markdown(f"**{name}:** :{color}[{label}] — confidence: {conf:.1%}")
            st.progress(float(conf))
        st.caption(f"Statement analysed: {user_input[:100]}")

st.divider()
st.caption("LIAR dataset (Wang, 2017) | Coventry University MSc Dissertation 2026")
"""

os.makedirs('/content/misinformation_app/src', exist_ok=True)
with open('/content/misinformation_app/src/app.py', 'w') as f:
    f.write(app)

print("✅ app.py created")

✅ app.py created


In [32]:
import subprocess, time, os
from pyngrok import conf, ngrok

# Paste your token here
NGROK_TOKEN = "3HMQucxXWvAYBUXdURK54aVCxQw_7fYWNMhaKoFiDrdF9FGkV"

# Set token
conf.get_default().auth_token = NGROK_TOKEN

os.chdir('/content/misinformation_app')

# Start Streamlit
process = subprocess.Popen([
    'streamlit', 'run', 'src/app.py',
    '--server.port=8501',
    '--server.headless=true',
    '--server.enableCORS=false'
])

time.sleep(8)

# Get public URL
public_url = ngrok.connect(8501)
print("=" * 50)
print("✅ APP IS LIVE")
print(f"🌐 Open: {public_url}")
print("=" * 50)

✅ APP IS LIVE
🌐 Open: NgrokTunnel: "https://designer-unaudited-kitchen.ngrok-free.dev" -> "http://localhost:8501"


In [1]:
import os

# Check all possible locations
locations = [
    '/content/best_roberta_liar',
    '/content/drive/MyDrive/dissertation/models/roberta_3epochs',
    '/content/drive/MyDrive/dissertation/models',
]

found = False
for loc in locations:
    if os.path.exists(loc):
        print(f"✅ Found at: {loc}")
        print(f"   Files: {os.listdir(loc)}")
        found = True

if not found:
    print("❌ Model not found anywhere")
    print("You need to retrain — takes 20 mins")

❌ Model not found anywhere
You need to retrain — takes 20 mins
